# Big Data Lab 1 - Ingest → Validate → Store (Parquet "bronze")

In this first lab, you'll build a small but realistic data pipeline from scratch. We start by downloading a real public dataset - the NYC taxi trip records - and treating it the way a data engineer would: we inspect its schema, take a careful first look without loading everything into memory, and then produce a compact data-quality report to see where the data can break our assumptions. From there, we create a “bronze” version of the dataset: minimally cleaned, standardized, and stored in a form that's meant to be reused by later processing steps. Finally, we write the result as a partitioned Parquet dataset and query it directly to confirm that what we produced is correct.

By the end, you ought to have a durable data artifact and the basic workflow instincts that make big data work reliable.

## What you’ll do (end-to-end)
1. Download a real public dataset (NYC Taxi trip records, Parquet).
2. Inspect schema + take a quick look at the data.
3. Run a **data-quality validation report** (nulls, ranges, logical constraints).
4. Create a cleaned **bronze** dataset and store it as **partitioned Parquet**.
5. Query the produced Parquet dataset to confirm it’s correct.

## Why this matters
Big-data work is often less about “one huge model” and more about building **reliable pipelines**:
- ingest from a source,
- check quality early (before errors propagate),
- store in efficient formats (Parquet),
- partition wisely for faster future queries.

---

## Evaluation
**50% correctness + 50% explanation**

That’s a strong approach for Big Data labs:
- **Correctness (50%)**: outputs exist, checks pass, queries match expected results, and the pipeline can be re-run.
- **Explanation (50%)**: student can explain *why* choices were made (format, partition key, validation rules), interpret results, and answer “what if” questions.

## Setup (run once)

We’ll use:
- **DuckDB** for fast SQL over Parquet (no server setup).
- **PyArrow** for writing partitioned Parquet datasets.

In [ ]:
!pip -q install duckdb pyarrow

import os
from pathlib import Path
import duckdb
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd

print("duckdb", duckdb.__version__)
print("pyarrow", pa.__version__)
print("pandas", pd.__version__)

## Optional: Persist outputs in Google Drive

Colab runtimes can reset. To avoid losing outputs, store your datasets in Drive.

If you don't want to use Drive, set `USE_DRIVE = False` and everything will stay in Colab’s local storage.

In [ ]:
USE_DRIVE = False  # <- change to True to persist outputs in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/BDLab/Lab1")
else:
    BASE_DIR = Path("/content/BDLab/Lab1")

RAW_DIR = BASE_DIR / "raw"
BRONZE_DIR = BASE_DIR / "bronze_partitioned"
RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR, RAW_DIR, BRONZE_DIR

## 1) Download a real dataset (NYC Taxi trip data)

We’ll use **Yellow Taxi trip records** published by NYC TLC.

Dataset files follow a predictable pattern like:

`https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_YYYY-MM.parquet`

We’ll download one month (typically ~50–70MB for many months).

In [ ]:
# Choose a month (keep it simple for Lab 1)
YEAR = 2024
MONTH = 1  # January

url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{YEAR}-{MONTH:02d}.parquet"
raw_path = RAW_DIR / f"yellow_tripdata_{YEAR}-{MONTH:02d}.parquet"

if not raw_path.exists():
    print("Downloading:", url)
    !wget -q "{url}" -O "{raw_path}"
else:
    print("Already downloaded:", raw_path)

raw_path, raw_path.stat().st_size / (1024*1024)

## 2) Inspect schema & preview rows (without loading everything)

A classic Big Data habit:
- **don’t** `read_parquet()` into memory blindly,
- **do** query/inspect selectively.

DuckDB can query Parquet directly.

In [ ]:
con = duckdb.connect()

# Show columns and types
schema_df = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{raw_path.as_posix()}')"
).df()
schema_df

In [ ]:
# Preview a few rows
sample_df = con.execute(f'''
SELECT *
FROM read_parquet('{raw_path.as_posix()}')
LIMIT 5
''').df()

sample_df

### Tiny exercise 1 (2–5 minutes)
In the next code cell:
1. Print the number of rows in the file.
2. Print the minimum and maximum pickup datetime.

*Hint:* use DuckDB SQL: `SELECT COUNT(*) ...` and `MIN()/MAX()`.

In [ ]:
# Tiny exercise 1
# Task:
# 1) Print the number of rows in the file.
# 2) Print the minimum and maximum pickup datetime.
#
# Hints:
# - Use DuckDB SQL: SELECT COUNT(*) ...
# - Use MIN()/MAX() on tpep_pickup_datetime
# - Use con.execute(...).fetchone() for scalar results

# TODO: write your SQL queries here.

raise NotImplementedError(
    "Exercise 1: compute row count and min/max pickup datetime using DuckDB SQL."
)


## 3) Data-quality validation report

We’ll implement a small set of practical checks. The goal is not “perfect data”, but **early detection** of issues.

Common first checks:
- Nulls in key columns
- Non-negative numeric fields (distance, fare, tip…)
- Logical constraints (dropoff time after pickup time)
- Reasonable ranges (passenger_count in a plausible range)

We’ll compute a quick report with DuckDB so we don’t have to load everything into pandas.

In [ ]:
parquet_ref = f"read_parquet('{raw_path.as_posix()}')"

# We'll build a validation report as a list of (check_name, failing_rows)
checks = []

# 1) Null checks
checks.append(("null_pickup_datetime", f"SELECT COUNT(*) FROM {parquet_ref} WHERE tpep_pickup_datetime IS NULL"))
checks.append(("null_dropoff_datetime", f"SELECT COUNT(*) FROM {parquet_ref} WHERE tpep_dropoff_datetime IS NULL"))
checks.append(("null_passenger_count", f"SELECT COUNT(*) FROM {parquet_ref} WHERE passenger_count IS NULL"))
checks.append(("null_trip_distance", f"SELECT COUNT(*) FROM {parquet_ref} WHERE trip_distance IS NULL"))

# 2) Basic numeric sanity
checks.append(("negative_trip_distance", f"SELECT COUNT(*) FROM {parquet_ref} WHERE trip_distance < 0"))
checks.append(("negative_fare_amount", f"SELECT COUNT(*) FROM {parquet_ref} WHERE fare_amount < 0"))
checks.append(("negative_total_amount", f"SELECT COUNT(*) FROM {parquet_ref} WHERE total_amount < 0"))

# 3) Logical constraints
checks.append(("dropoff_before_pickup", f'''
SELECT COUNT(*) FROM {parquet_ref}
WHERE tpep_dropoff_datetime < tpep_pickup_datetime
'''))

# 4) Plausible ranges (tune as you like)
checks.append(("passenger_count_out_of_range", f'''
SELECT COUNT(*) FROM {parquet_ref}
WHERE passenger_count < 0 OR passenger_count > 8
'''))

report = []
for name, sql in checks:
    failing = con.execute(sql).fetchone()[0]
    report.append({"check": name, "failing_rows": int(failing)})

report_df = pd.DataFrame(report).sort_values("failing_rows", ascending=False)
report_df

### Interpreting the report (talking points for oral evaluation)
- Which checks show failures? Are they “small noise” or “systematic”?
- If a check fails, what are your options?
  - drop invalid rows,
  - fix/impute,
  - quarantine for later,
  - or accept and document.

There isn’t always one correct answer — but you must justify your choice.

### Tiny exercise 2 (5–10 minutes)
Add **one more check** that you think is useful and compute the number of failing rows.

Examples:
- `tip_amount < 0`
- `trip_distance == 0 AND total_amount > 0` (weird pricing)
- `fare_amount == 0 AND trip_distance > 0`

Put your check into the cell below as another SQL query and print the failing count.

In [ ]:
# Tiny exercise 2
# Add ONE additional data-quality check you think is useful and print the failing row count.
#
# Hints:
# - Use the already-defined `parquet_ref` (a read_parquet(...) reference).
# - Example checks:
#   * tip_amount < 0
#   * trip_distance = 0 AND total_amount > 0
#   * fare_amount = 0 AND trip_distance > 0
# - If you're not sure a column exists, you can inspect `schema_df`.

# TODO: write one SQL query that returns COUNT(*) of failing rows.
# Example shape:
# sql = f"SELECT COUNT(*) FROM {parquet_ref} WHERE ..."
# failing = con.execute(sql).fetchone()[0]
# print("Failing rows:", failing)

raise NotImplementedError(
    "Exercise 2: add one more validation check and compute failing rows."
)


## 4) Build a cleaned "bronze" dataset (partitioned Parquet)

We’ll create a *bronze* dataset:
- same core fields,
- basic filtering of obviously invalid rows,
- add a partition column: `pickup_date`.

**Why partition?**  
If your future queries filter by date, partitions allow scanning less data.

### Design choice for this lab
To keep runtime & output size reasonable, we’ll export **only the first 7 days of the month**.
(Your assignment at the end asks you to extend this.)

In [ ]:
# We'll select a week of data (first 7 days of the month)
# Compute pickup_date and filter on it
start_date = f"{YEAR}-{MONTH:02d}-01"
end_date   = f"{YEAR}-{MONTH:02d}-08"  # exclusive

clean_sql = f'''
SELECT
  -- keep a practical subset of columns (you can expand later)
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  PULocationID,
  DOLocationID,
  passenger_count,
  trip_distance,
  fare_amount,
  tip_amount,
  total_amount,
  CAST(tpep_pickup_datetime AS DATE) AS pickup_date
FROM {parquet_ref}
WHERE
  tpep_pickup_datetime >= TIMESTAMP '{start_date}'
  AND tpep_pickup_datetime <  TIMESTAMP '{end_date}'
  AND tpep_dropoff_datetime >= tpep_pickup_datetime
  AND trip_distance >= 0
  AND fare_amount >= 0
  AND total_amount >= 0
'''

# Materialize into an Arrow table (efficient bridge to PyArrow)
arrow_tbl = con.execute(clean_sql).fetch_arrow_table()
arrow_tbl.num_rows, arrow_tbl.num_columns

In [ ]:
# Write as a partitioned Parquet dataset: bronze_partitioned/pickup_date=YYYY-MM-DD/part-*.parquet
# We overwrite the directory each run for idempotency (safe reruns).

import shutil
import pyarrow as pa
import pyarrow.dataset as ds
if BRONZE_DIR.exists():
    shutil.rmtree(BRONZE_DIR)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)



hive_part = ds.partitioning(
    pa.schema([("pickup_date", pa.date32())]),
    flavor="hive"
)

ds.write_dataset(
    data=arrow_tbl,
    base_dir=str(BRONZE_DIR),
    format="parquet",
    partitioning=hive_part,
    existing_data_behavior="overwrite_or_ignore"
)

# Show resulting folder structure (first few files)
paths = sorted([p.as_posix() for p in BRONZE_DIR.rglob("*.parquet")])
paths[:5], len(paths)




## 5) Query the produced Parquet dataset (sanity check)

Now you have an actual "bronze" dataset you can reuse later.
Let’s query it directly from disk.

In [ ]:
bronze_ref = f"read_parquet('{(BRONZE_DIR / '**/*.parquet').as_posix()}', hive_partitioning=1)"

# How many rows per day?
per_day = con.execute(f'''
SELECT pickup_date, COUNT(*) AS trips
FROM {bronze_ref}
GROUP BY 1
ORDER BY 1
''').df()
per_day


### Tiny exercise 3 (5–10 minutes)
Write a query that computes:
- average trip distance per day
- average total amount per day

Put your SQL in the next cell and display the result as a DataFrame.

In [ ]:
# Tiny exercise 3
# Write a query that computes:
# - average trip distance per day
# - average total amount per day
#
# Hints:
# - Use the already-defined `bronze_ref` (partitioned Parquet with hive_partitioning=1).
# - Use AVG(...), GROUP BY pickup_date, ORDER BY pickup_date.
# - Return a DataFrame with .df().

# TODO: write your SQL and display the result DataFrame.

raise NotImplementedError(
    "Exercise 3: compute per-day averages with DuckDB SQL over partitioned Parquet."
)


# Mini-assignment (end of lab)

This is designed for manual grading + live explanation.

## Assignment A — Extend to another time range
1. Download **another month** (e.g., Feb 2024).
2. Produce a bronze dataset for the **first 7 days** of that month too.
3. Store it under a separate folder, e.g. `bronze_partitioned_2024_02/`.

## Assignment B — Answer with SQL
Using DuckDB over your partitioned Parquet, answer:
1. Which day has the highest number of trips?
2. What is the median (`quantile_cont`) total amount per day?
3. For `passenger_count >= 3`, what is the average trip distance per day?

## Assignment C — Oral explanation prompts (prepare short answers)
- Why is Parquet better than CSV for analytics?
- Why partition by date here? When would it be a bad idea?
- If the validation report shows failures, what would you do in a production pipeline?

---
### Skeleton (optional)
The cell below is a *non-executing* skeleton you can copy and adapt.

In [ ]:
if False:
    YEAR2, MONTH2 = 2024, 2
    url2 = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{YEAR2}-{MONTH2:02d}.parquet"
    raw_path2 = RAW_DIR / f"yellow_tripdata_{YEAR2}-{MONTH2:02d}.parquet"

    # 1) download
    # !wget -q "{url2}" -O "{raw_path2}"

    # 2) clean+export (adapt clean_sql with new dates)
    # 3) query answers with duckdb
    pass

## What you should have at the end
- A raw Parquet file downloaded in `raw/`
- A partitioned bronze dataset in `bronze_partitioned/`
- A validation report table (even if some checks fail)
- A few DuckDB queries that run directly on Parquet

If you can **run it twice** and get the same result, you’re already practicing good pipeline discipline.

# References (optional)

Use these only if you get stuck; everything needed for the lab is in the notebook.

- DuckDB docs (start here): https://duckdb.org/docs/
- DuckDB Parquet overview + `read_parquet`: https://duckdb.org/docs/data/parquet/overview.html
- DuckDB aggregate functions (for `quantile_cont`/median): https://duckdb.org/docs/sql/functions/aggregates.html
- PyArrow dataset writer (`write_dataset`) + partitioning: https://arrow.apache.org/docs/python/generated/pyarrow.dataset.write_dataset.html
